# FPL Transcript Data Generator

Set `gameweek` to the upcoming gameweek, add YouTube transcripts to `gw/gw{gameweek}/transcripts/`, then run this notebook top to bottom before generating a report.

The notebook groups each transcript into broad sections, retains the FPL analysis, resolves ambiguous player names, and extracts atomic player recommendations with an LLM.

The report-ready extraction artifact is written to:

```text
gw/gw{gameweek}/data/gw_{gameweek}_transcript_extractions.json
```

Each recommendation contains a canonical player name where resolution supports one, a 1-5 stance, the full interpreted sentiment, interpretation confidence, a source-faithful supporting quote, and its transcript source. Responses are cached per pass under `gw/gw{gameweek}/transcript_cache/`, so completed work is reused when the notebook is run again.

In [1]:
import os

# Ensure all relative paths (gw/, lookup/, reports/) resolve from the project root,
# regardless of whether the notebook is opened from notebooks/ or notebooks/transcripts_processor/.
cwd = os.getcwd()
if os.path.basename(cwd) == "transcripts_processor":
    os.chdir("../..")

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/augustustranter/Documents/Programming/Projects/fpl/fpl_agent


In [2]:
# This variable indicates the gameweek for which we are generating a report.
# E.g. if gameweek = 3, transcripts in gw/gw3/transcripts/ are analysed.
gameweek = 5

In [3]:
import json
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

data_dir = PROJECT_ROOT / "gw" / f"gw{gameweek}" / "data"
transcript_dir = PROJECT_ROOT / "gw" / f"gw{gameweek}" / "transcripts"
transcript_cache_dir = PROJECT_ROOT / "gw" / f"gw{gameweek}" / "transcript_cache"
segmentation_cache_dir = transcript_cache_dir / "segmentation_cache"
name_resolution_cache_dir = transcript_cache_dir / "name_resolution_cache"
extraction_cache_dir = transcript_cache_dir / "extraction_cache"
aggregate_path = data_dir / f"gw_{gameweek}_aggregated_player_form.json"
element_lookup_path = PROJECT_ROOT / "lookup" / "element_lookup.json"
extractions_output_path = data_dir / f"gw_{gameweek}_transcript_extractions.json"
if not aggregate_path.exists():
    raise FileNotFoundError(
        f"Missing {aggregate_path.relative_to(PROJECT_ROOT)}. Run statistics_generator.ipynb for GW{gameweek} first."
    )
if not element_lookup_path.exists():
    raise FileNotFoundError(f"Missing {element_lookup_path.relative_to(PROJECT_ROOT)}")

with aggregate_path.open(encoding="utf-8") as file:
    players = json.load(file)
with element_lookup_path.open(encoding="utf-8") as file:
    element_lookup = json.load(file)

player_names = {player["name"]: int(player_id) for player_id, player in players.items()}
canonical_player_names = {player["name"]: int(player_id) for player_id, player in element_lookup.items()}
canonical_player_id_to_name = {player_id: name for name, player_id in canonical_player_names.items()}

print(f"Loaded {len(player_names)} players from {aggregate_path.relative_to(PROJECT_ROOT)}")
print(
    f"Loaded {len(canonical_player_names)} canonical player names from "
    f"{element_lookup_path.relative_to(PROJECT_ROOT)}"
)
print(f"Segmentation cache: {segmentation_cache_dir.relative_to(PROJECT_ROOT)}")
print(f"Name resolution cache: {name_resolution_cache_dir.relative_to(PROJECT_ROOT)}")
print(f"Extraction cache: {extraction_cache_dir.relative_to(PROJECT_ROOT)}")
print(f"Transcript extractions output: {extractions_output_path.relative_to(PROJECT_ROOT)}")

Loaded 659 players from gw/gw5/data/gw_5_aggregated_player_form.json
Loaded 659 canonical player names from lookup/element_lookup.json
Segmentation cache: gw/gw5/transcript_cache/segmentation_cache
Name resolution cache: gw/gw5/transcript_cache/name_resolution_cache
Extraction cache: gw/gw5/transcript_cache/extraction_cache
Transcript extractions output: gw/gw5/data/gw_5_transcript_extractions.json


# 1. SEGMENTATION

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib

transcript_segmentation = importlib.import_module("notebooks.transcripts_processor.1_segmentation")
transcript_segmentation = importlib.reload(transcript_segmentation)
build_analysis_text = transcript_segmentation.build_analysis_text
collect_transcripts = transcript_segmentation.collect_transcripts
segment_transcript = transcript_segmentation.segment_transcript
MAX_CONCURRENT_TRANSCRIPTS = transcript_segmentation.MAX_CONCURRENT_TRANSCRIPTS
SEGMENTATION_TIMEOUT_SECONDS = transcript_segmentation.SEGMENTATION_TIMEOUT_SECONDS
SEGMENTATION_MAX_RETRIES = transcript_segmentation.SEGMENTATION_MAX_RETRIES
KEEP_SEGMENT_TYPES = transcript_segmentation.KEEP_SEGMENT_TYPES

model = "gpt-4.1-mini"

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(
    api_key=api_key,
    timeout=SEGMENTATION_TIMEOUT_SECONDS,
    max_retries=SEGMENTATION_MAX_RETRIES,
 )

transcripts = collect_transcripts(transcript_dir) if transcript_dir.exists() else []
sources = [transcript["source"] for transcript in transcripts]
print(f"Found {len(transcripts)} transcript(s) in {transcript_dir.relative_to(PROJECT_ROOT)}")

segment_records: list[dict] = []
failed_segment_sources: list[dict[str, str]] = []
if transcripts:
    max_workers = min(MAX_CONCURRENT_TRANSCRIPTS, len(transcripts))
    print(
        f"Segmenting with {max_workers} worker(s), a {SEGMENTATION_TIMEOUT_SECONDS}s request timeout, "
        f"and {SEGMENTATION_MAX_RETRIES} automatic retries"
    )
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                segment_transcript,
                client,
                model,
                segmentation_cache_dir,
                transcript["text"],
                gameweek,
            ): transcript
            for transcript in transcripts
        }
        for completed, future in enumerate(as_completed(futures), start=1):
            transcript = futures[future]
            source = transcript["source"]
            try:
                segments = future.result()
            except Exception as exc:
                failed_segment_sources.append(
                    {"source": source, "error": f"{type(exc).__name__}: {exc}"}
                )
                print(f"[{completed}/{len(transcripts)}] FAILED {source}: {type(exc).__name__}: {exc}")
                continue

            analysis_text = build_analysis_text(segments)
            segment_records.append(
                {
                    "source": source,
                    "segments": segments,
                    "analysis_text": analysis_text,
                }
            )
            retained_count = sum(segment.get("type") in KEEP_SEGMENT_TYPES for segment in segments)
            print(
                f"[{completed}/{len(transcripts)}] Completed {source}: "
                f"{len(segments)} section(s), {retained_count} retained"
            )
    segment_records.sort(key=lambda record: sources.index(record["source"]))

successful_sources = [record["source"] for record in segment_records]
segment_output = {
    "sources": successful_sources,
    "segment_records": segment_records,
    "failed_sources": failed_segment_sources,
}
print(
    f"Prepared {len(segment_records)} segmented transcript(s) in memory; "
    f"{len(failed_segment_sources)} failed"
 )
if failed_segment_sources:
    print("Failed sources can be retried by running this cell again; completed sources will use the cache:")
    for failure in failed_segment_sources:
        print(f"- {failure['source']}: {failure['error']}")
print(f"Segmentation cache used: {segmentation_cache_dir.relative_to(PROJECT_ROOT)}")

Found 72 transcript(s) in gw/gw5/transcripts
Segmenting with 2 worker(s), a 90s request timeout, and 5 automatic retries
[1/72] Completed -kAKyG1vM1k: 12 section(s), 11 retained
[2/72] Completed 0pl8M7GNavc: 12 section(s), 11 retained
[3/72] Completed 1j0h5W5_YfE: 7 section(s), 7 retained
[4/72] Completed 1OzRIdPoXAg: 3 section(s), 3 retained
[5/72] Completed 1pThhv6Ni_w: 1 section(s), 1 retained
[6/72] Completed 2VEc1CcE4CE: 4 section(s), 4 retained
[7/72] Completed 7iRAYKtQ0ZU: 2 section(s), 1 retained
[8/72] Completed 66PRV226E1k: 17 section(s), 16 retained
[9/72] Completed 8vc8yng3cak: 3 section(s), 2 retained
[10/72] Completed 9EtCsG2xjMc: 3 section(s), 2 retained
[11/72] Completed BBOyant7K38: 4 section(s), 4 retained
[12/72] Completed CTyv6aa_wxQ: 2 section(s), 1 retained
[13/72] Completed CoQ1XXWyKIM: 2 section(s), 1 retained
[14/72] Completed Dl0thXyMTwM: 2 section(s), 1 retained
[15/72] Completed E3Nvf7DqZqw: 11 section(s), 11 retained
[16/72] Completed FzxCYX7EY5s: 4 section

# NAME CORRECTION/RESOLUTION

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed

transcript_name_resolution = importlib.import_module("notebooks.transcripts_processor.2_name_resolution")
transcript_name_resolution = importlib.reload(transcript_name_resolution)
build_name_resolution_helpers = transcript_name_resolution.build_name_resolution_helpers
load_name_aliases = transcript_name_resolution.load_name_aliases
MAX_CONCURRENT_NAME_RESOLUTION = transcript_name_resolution.MAX_CONCURRENT_NAME_RESOLUTION

model = "gpt-4.1-mini"

NAME_ALIASES = load_name_aliases(PROJECT_ROOT)
PLAYER_METADATA = {
    int(player_id): {
        "team": str(player.get("team", "")),
        "position": int(player.get("position", 0) or 0),
    }
    for player_id, player in players.items()
}
name_resolution = build_name_resolution_helpers(
    name_aliases=NAME_ALIASES,
    canonical_player_names=canonical_player_names,
    canonical_player_id_to_name=canonical_player_id_to_name,
    player_metadata=PLAYER_METADATA,
)
resolve_transcript_names = name_resolution.resolve_transcript_names

segment_records = globals().get("segment_records")
if segment_records is None:
    raise RuntimeError("Run the segment pass first.")

print(f"Loaded {len(segment_records)} segmented transcript(s) from memory")

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(api_key=api_key, max_retries=5)

name_corrections_by_source: dict[str, list[dict]] = {}
resolution_inputs = [
    record
    for record in segment_records
    if str(record.get("analysis_text", "")).strip()
]
skipped_inputs = [
    record
    for record in segment_records
    if not str(record.get("analysis_text", "")).strip()
]
print(
    f"Name resolution will process {len(resolution_inputs)} transcript(s) and skip {len(skipped_inputs)} "
    f"without retained analysis text"
)
if resolution_inputs:
    max_workers = min(MAX_CONCURRENT_NAME_RESOLUTION, len(resolution_inputs))
    print(f"Resolving names with {max_workers} worker(s); cache: {name_resolution_cache_dir.relative_to(PROJECT_ROOT)}")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                resolve_transcript_names,
                client,
                model,
                name_resolution_cache_dir,
                str(record.get("source", "")),
                str(record.get("analysis_text", "")).strip(),
            ): record
            for record in resolution_inputs
        }
        for completed, future in enumerate(as_completed(futures), start=1):
            record = futures[future]
            source = str(record.get("source", ""))
            try:
                corrections = future.result()
            except Exception as exc:
                print(f"[{completed}/{len(resolution_inputs)}] FAILED {source}: {type(exc).__name__}: {exc}")
                raise
            name_corrections_by_source[source] = corrections
            correction_total = len(corrections)
            resolved_total = sum(correction["resolution_status"] == "resolved" for correction in corrections)
            unresolved_total = correction_total - resolved_total
            print(
                f"[{completed}/{len(resolution_inputs)}] Completed {source}: "
                f"{correction_total} correction(s), {resolved_total} resolved, {unresolved_total} unresolved"
            )

correction_count = sum(len(corrections) for corrections in name_corrections_by_source.values())
resolved_count = sum(
    correction["resolution_status"] == "resolved"
    for corrections in name_corrections_by_source.values()
    for correction in corrections
)
name_corrections_output_path = data_dir / f"gw_{gameweek}_name_corrections.json"
name_corrections_output = {
    "sources": segment_output.get("sources", []),
    "count": correction_count,
    "resolved_count": resolved_count,
    "skipped_sources": [str(record.get("source", "")) for record in skipped_inputs],
    "name_corrections_by_source": name_corrections_by_source,
}
name_corrections_output_path.write_text(
    json.dumps(name_corrections_output, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(
    f"Name corrections written to {name_corrections_output_path.relative_to(PROJECT_ROOT)} "
    f"({correction_count} total, {resolved_count} resolved)"
)
print(f"Skipped {len(skipped_inputs)} transcript(s) with no retained analysis text")
print(f"Name resolution cache used: {name_resolution_cache_dir.relative_to(PROJECT_ROOT)}")

Loaded 72 segmented transcript(s) from memory
Name resolution will process 72 transcript(s) and skip 0 without retained analysis text
Resolving names with 2 worker(s); cache: gw/gw5/transcript_cache/name_resolution_cache
[1/72] Completed 0pl8M7GNavc: 85 correction(s), 74 resolved, 11 unresolved
[2/72] Completed -kAKyG1vM1k: 91 correction(s), 84 resolved, 7 unresolved
[3/72] Completed 1j0h5W5_YfE: 50 correction(s), 47 resolved, 3 unresolved
[4/72] Completed 1OzRIdPoXAg: 30 correction(s), 27 resolved, 3 unresolved
[5/72] Completed 1pThhv6Ni_w: 45 correction(s), 41 resolved, 4 unresolved
[6/72] Completed 2VEc1CcE4CE: 31 correction(s), 31 resolved, 0 unresolved
[7/72] Completed 66PRV226E1k: 60 correction(s), 56 resolved, 4 unresolved
[8/72] Completed 7iRAYKtQ0ZU: 26 correction(s), 25 resolved, 1 unresolved
[9/72] Completed 8vc8yng3cak: 24 correction(s), 24 resolved, 0 unresolved
[10/72] Completed 9EtCsG2xjMc: 27 correction(s), 24 resolved, 3 unresolved
[11/72] Completed BBOyant7K38: 29 cor

# 3. RETROSPECTIVE FILTER

In [10]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed

transcript_selection = importlib.import_module("notebooks.transcripts_processor.3_selection_filter")
transcript_selection = importlib.reload(transcript_selection)
filter_transcript_analysis = transcript_selection.filter_transcript_analysis
write_transcript_selection_output = transcript_selection.write_transcript_selection_output
MAX_CONCURRENT_SELECTION_FILTERS = transcript_selection.MAX_CONCURRENT_SELECTION_FILTERS

model = "gpt-4.1-mini"

segment_records = globals().get("segment_records")
if segment_records is None:
    raise RuntimeError("Run the segment pass first.")

name_corrections_by_source = globals().get("name_corrections_by_source")
if name_corrections_by_source is None:
    raise RuntimeError("Run the name pass first.")

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(api_key=api_key, max_retries=5)

selection_cache_dir = transcript_cache_dir / "selection_cache"
selection_output_path = data_dir / f"gw_{gameweek}_transcript_selection.json"

selection_inputs = [
    record
    for record in segment_records
    if str(record.get("analysis_text", "")).strip()
]
skipped_inputs = [
    record
    for record in segment_records
    if not str(record.get("analysis_text", "")).strip()
]
print(
    f"Selection filtering will process {len(selection_inputs)} transcript(s) and skip {len(skipped_inputs)} "
    f"without retained analysis text"
)

selection_by_source: dict[str, dict] = {}
failed_selection_sources: list[dict[str, str]] = []
if selection_inputs:
    max_workers = min(MAX_CONCURRENT_SELECTION_FILTERS, len(selection_inputs))
    print(
        f"Filtering retrospective-only sections with {max_workers} worker(s); cache: "
        f"{selection_cache_dir.relative_to(PROJECT_ROOT)}"
    )
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                filter_transcript_analysis,
                client,
                model,
                selection_cache_dir,
                str(record.get("source", "")),
                str(record.get("analysis_text", "")).strip(),
            ): record
            for record in selection_inputs
        }
        for completed, future in enumerate(as_completed(futures), start=1):
            record = futures[future]
            source = str(record.get("source", ""))
            try:
                selection_by_source[source] = future.result()
            except Exception as exc:
                failed_selection_sources.append({"source": source, "error": str(exc)})
                print(f"[{completed}/{len(futures)}] Failed {source}: {exc}")
                raise
            else:
                retained_count = selection_by_source[source]["counts"]["retained_sections"]
                removed_count = selection_by_source[source]["counts"]["removed_sections"]
                print(
                    f"[{completed}/{len(futures)}] Completed {source}: "
                    f"{retained_count} retained, {removed_count} removed"
                )

selection_records = [
    selection_by_source[str(record.get("source", ""))]
    for record in selection_inputs
    if str(record.get("source", "")) in selection_by_source
]
filtered_segment_records = []
for record in selection_inputs:
    source = str(record.get("source", ""))
    selection = selection_by_source.get(source)
    if not selection:
        continue
    filtered_analysis_text = str(selection.get("filtered_analysis_text", "")).strip()
    filtered_segment_records.append(
        {
            **record,
            "raw_analysis_text": str(record.get("analysis_text", "")).strip(),
            "analysis_text": filtered_analysis_text,
            "filtered_analysis_text": filtered_analysis_text,
            "retained_section_count": selection["counts"]["retained_sections"],
            "removed_section_count": selection["counts"]["removed_sections"],
            "selection": selection,
        }
    )

selection_output = write_transcript_selection_output(
    selection_output_path,
    sources=segment_output.get("sources", []),
    selections=selection_records,
)

print(
    f"Selection filter retained {sum(item['counts']['retained_sections'] for item in selection_records)} sections "
    f"across {len(selection_records)} source(s); {len(failed_selection_sources)} failed"
)
print(
    f"Transcript selection output written to {selection_output_path.relative_to(PROJECT_ROOT)} "
    f"({selection_output['counts']['sources']} source(s))"
)
print(f"Selection cache used: {selection_cache_dir.relative_to(PROJECT_ROOT)}")

Selection filtering will process 72 transcript(s) and skip 0 without retained analysis text
Filtering retrospective-only sections with 6 worker(s); cache: gw/gw5/transcript_cache/selection_cache
[1/72] Completed 1j0h5W5_YfE: 6 retained, 1 removed
[2/72] Completed 0pl8M7GNavc: 9 retained, 2 removed
[3/72] Completed 1OzRIdPoXAg: 2 retained, 1 removed
[4/72] Completed 2VEc1CcE4CE: 2 retained, 2 removed
[5/72] Completed -kAKyG1vM1k: 11 retained, 0 removed
[6/72] Completed 1pThhv6Ni_w: 1 retained, 0 removed
[7/72] Completed 66PRV226E1k: 16 retained, 0 removed
[8/72] Completed 9EtCsG2xjMc: 1 retained, 1 removed
[9/72] Completed CTyv6aa_wxQ: 1 retained, 0 removed
[10/72] Completed 8vc8yng3cak: 2 retained, 0 removed
[11/72] Completed 7iRAYKtQ0ZU: 1 retained, 0 removed
[12/72] Completed BBOyant7K38: 3 retained, 1 removed
[13/72] Completed Dl0thXyMTwM: 1 retained, 0 removed
[14/72] Completed FzxCYX7EY5s: 2 retained, 1 removed
[15/72] Completed GrAO2UfXTJs: 4 retained, 0 removed
[16/72] Completed

# 4. EXTRACTION

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed

transcript_extraction = importlib.import_module("notebooks.transcripts_processor.4_extraction")
transcript_extraction = importlib.reload(transcript_extraction)
extract_from_transcript = transcript_extraction.extract_from_transcript
write_transcript_extractions_output = transcript_extraction.write_transcript_extractions_output
MAX_CONCURRENT_EXTRACTIONS = transcript_extraction.MAX_CONCURRENT_EXTRACTIONS

model = "gpt-4.1-mini"

filtered_segment_records = globals().get("filtered_segment_records")
if filtered_segment_records is None:
    filtered_segment_records = globals().get("segment_records")
if filtered_segment_records is None:
    raise RuntimeError("Run the segment and filter passes first.")

name_corrections_by_source = globals().get("name_corrections_by_source")
if name_corrections_by_source is None:
    raise RuntimeError("Run the name pass first.")

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env.")

client = OpenAI(api_key=api_key, max_retries=5)

extraction_inputs = [
    {
        "source": str(record.get("source", "")),
        "analysis_text": str(record.get("analysis_text", "")).strip(),
    }
    for record in filtered_segment_records
    if str(record.get("analysis_text", "")).strip()
]
parsed_by_source: dict[str, dict] = {}
failed_extraction_sources: list[dict[str, str]] = []

if extraction_inputs:
    max_workers = min(MAX_CONCURRENT_EXTRACTIONS, len(extraction_inputs))
    print(f"Extracting player recommendations with {max_workers} worker(s)")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                extract_from_transcript,
                client,
                model,
                extraction_cache_dir,
                item["source"],
                item["analysis_text"],
                name_corrections_by_source.get(item["source"], []),
            ): item["source"]
            for item in extraction_inputs
        }
        for completed, future in enumerate(as_completed(futures), start=1):
            source = futures[future]
            try:
                parsed_by_source[source] = future.result()
            except Exception as exc:
                failed_extraction_sources.append({"source": source, "error": str(exc)})
                print(f"[{completed}/{len(futures)}] Failed {source}: {exc}")
            else:
                count = len(parsed_by_source[source].get("recommendations", []))
                print(f"[{completed}/{len(futures)}] Completed {source}: {count} recommendation(s)")

recommendations: list[dict] = []
for item in extraction_inputs:
    source = item["source"]
    for recommendation in parsed_by_source.get(source, {}).get("recommendations", []):
        recommendations.append({**recommendation, "source": source})

transcript_extractions_output = write_transcript_extractions_output(
    extractions_output_path,
    sources=segment_output.get("sources", []),
    recommendations=recommendations,
)

print(
    f"Extracted {len(recommendations)} player recommendations from "
    f"{len(parsed_by_source)} source(s); {len(failed_extraction_sources)} failed"
)
print(
    f"Transcript extractions written to {extractions_output_path.relative_to(PROJECT_ROOT)} "
    f"({transcript_extractions_output['counts']['sources']} source(s))"
)
print(f"Extraction cache used: {extraction_cache_dir.relative_to(PROJECT_ROOT)}")

Extracting player recommendations with 6 worker(s)
[1/72] Completed 1OzRIdPoXAg: 11 recommendation(s)
[2/72] Completed 2VEc1CcE4CE: 7 recommendation(s)
[3/72] Completed 1pThhv6Ni_w: 8 recommendation(s)
[4/72] Completed -kAKyG1vM1k: 11 recommendation(s)
[5/72] Completed 0pl8M7GNavc: 17 recommendation(s)
[6/72] Completed 66PRV226E1k: 20 recommendation(s)
[7/72] Completed 1j0h5W5_YfE: 23 recommendation(s)
[8/72] Completed 7iRAYKtQ0ZU: 17 recommendation(s)
[9/72] Completed 9EtCsG2xjMc: 12 recommendation(s)
[10/72] Completed 8vc8yng3cak: 18 recommendation(s)
[11/72] Completed CoQ1XXWyKIM: 5 recommendation(s)
[12/72] Completed E3Nvf7DqZqw: 18 recommendation(s)
[13/72] Completed BBOyant7K38: 16 recommendation(s)
[14/72] Completed CTyv6aa_wxQ: 6 recommendation(s)
[15/72] Completed G15nBTywM2U: 10 recommendation(s)
[16/72] Completed FzxCYX7EY5s: 16 recommendation(s)
[17/72] Completed Dl0thXyMTwM: 15 recommendation(s)
[18/72] Completed JKw1VQ1NPIg: 9 recommendation(s)
[19/72] Completed GGKjXYqmR